# 17 - Corporate Finance Readiness Advisor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/17-corporate-finance/corporate_finance_workbook.ipynb)

Assess a company's readiness for a capital markets transaction across five dimensions -- with a go/no-go gate per dimension.

**Harness focus:** Multi-dimension readiness scoring with go/no-go gate per dimension

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: Why dimensional gating beats a single score

A single readiness score of 7/10 can hide a fatal deal blocker. Dimensional gating prevents this by applying a pass/conditional/fail verdict to each dimension independently.

**Gate logic (strict):**

| Condition | overall_status |
|-----------|----------------|
| All five dimensions = pass | ready |
| One or more = conditional, none = fail | ready_with_conditions |
| Any dimension = fail | not_ready |

A strong financial score cannot compensate for a legal blocker. One fail = not_ready, period.

## Part 2: Schema -- gate field on every dimension

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class DimensionAssessment(BaseModel):
    dimension: Literal["governance", "financials", "market_position", "legal", "narrative"]
    score: int = Field(description="0-10 readiness score")
    gate: Literal["pass", "conditional", "fail"]
    strengths: List[str]
    blockers: List[str]
    remediation: List[str]

class ReadinessReport(BaseModel):
    company: Optional[str] = None
    transaction_type: Literal["ipo", "series_a", "series_b", "growth_equity", "pe_buyout"]
    overall_status: Literal["ready", "ready_with_conditions", "not_ready"]
    executive_summary: str
    dimensions: List[DimensionAssessment]
    critical_path: List[str]
    estimated_time_to_ready: str
    key_value_drivers: List[str]

print("Schema defined.")

## Part 3: System prompt -- encode the gate logic precisely

The overall_status rule must be encoded precisely in the system prompt.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

ADVISOR_SYSTEM = SystemMessage("""
You are a corporate finance advisor assessing transaction readiness.

Evaluate across FIVE dimensions in this order:
  1. governance      -- board, audit committee, financial controls
  2. financials      -- quality of earnings, audit status
  3. market_position -- TAM, moat, customer concentration, NRR
  4. legal           -- IP ownership, contracts, regulatory status
  5. narrative       -- investor story, team credibility

GATE per dimension:
  pass        = ready now
  conditional = fixable within 6 months
  fail        = structural blocker (>6 months)

OVERALL STATUS RULE:
  all pass               --> ready
  any conditional, no fail --> ready_with_conditions
  any fail               --> not_ready
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advisor = ADVISOR_SYSTEM | llm.with_structured_output(ReadinessReport)
print("Agent ready.")

## Part 4: Run on Volt Energy Technologies

In [ ]:
BRIEF = """
COMPANY: Volt Energy Technologies Ltd
Business: B2B energy management SaaS  |  Employees: 38

ARR: GBP 8.2m (+67% YoY)  |  Gross margin: 74%  |  NRR: 118%
Burn: GBP 380k/month  |  Runway: 14 months
FY2022 audited (4 months late); FY2023, FY2024 unaudited

Governance: CEO, CTO, CFO only. No NEDs, no audit committee.
Legal: 2 contractors built the core platform; no IP assignments signed.
Market: 210 customers; largest = 31% of ARR. 3 VC-backed competitors.
Transaction: Series B, GBP 15m
"""

report = advisor.invoke(HumanMessage(content="Company brief:\n\n" + BRIEF))
print(f"Status: {report.overall_status.upper()}")
print(f"Time to ready: {report.estimated_time_to_ready}")

## Part 5: Print the dimension scorecard

In [ ]:
GATE_LABEL = {"pass": "PASS", "conditional": "COND", "fail": "FAIL"}

print(report.executive_summary)
print("\nDIMENSION SCORECARD:")
for d in report.dimensions:
    print(f"\n  [{GATE_LABEL[d.gate]}] {d.dimension.upper()} -- {d.score}/10")
    if d.blockers:
        print(f"  Blockers: {'; '.join(d.blockers)}")
    if d.remediation:
        for step in d.remediation:
            print(f"    - {step}")

print("\nCRITICAL PATH:")
for i, action in enumerate(report.critical_path, 1):
    print(f"  {i}. {action}")

## Exercise: Add comparable_transactions field

Add `comparable_transactions: List[str]` to `ReadinessReport` -- 3 recent benchmark deals investors would compare this company against.

Format: `'CompanyName (year) -- round, amount, investor'`

**Starter:** Add the field below.

In [ ]:
# Starter: add comparable_transactions
class ReadinessReportV2(BaseModel):
    company: Optional[str] = None
    transaction_type: Literal["ipo", "series_a", "series_b", "growth_equity", "pe_buyout"]
    overall_status: Literal["ready", "ready_with_conditions", "not_ready"]
    executive_summary: str
    dimensions: List[DimensionAssessment]
    critical_path: List[str]
    estimated_time_to_ready: str
    key_value_drivers: List[str]
    comparable_transactions: List[str]  # NEW: 3 benchmark deals

# TODO: update ADVISOR_SYSTEM to suggest 3 comparable deals
# TODO: rebuild advisor_v2 = updated_system | llm.with_structured_output(ReadinessReportV2)
# TODO: run and print comparable_transactions

In [ ]:
# Answer key
ADVISOR_SYSTEM_V2 = SystemMessage("""
You are a corporate finance advisor assessing transaction readiness.

(Apply the same five-dimension gate logic as before.)

comparable_transactions: suggest 3 recent deals investors would use as benchmarks.
Format: 'CompanyName (year) -- round type, amount, lead investor'.
Pick deals in similar sector, geography, and stage.
""")

advisor_v2 = ADVISOR_SYSTEM_V2 | llm.with_structured_output(ReadinessReportV2)
report_v2 = advisor_v2.invoke(HumanMessage(content="Company brief:\n\n" + BRIEF))
print("Comparable transactions:")
for c in report_v2.comparable_transactions:
    print(f"  - {c}")

## What You Built

- A transaction readiness advisor with per-dimension gating
- `gate` field (pass/conditional/fail) blocking the overall status
- Strict gate logic: one fail = not_ready, cannot be overridden
- Critical path: ordered actions to reach 'ready' status

**Next steps:** Try example 18 (audience-targeted fundraising) using the same Volt Energy company brief.